In [ ]:
import sys
print(sys.version)


In [ ]:
!conda install -y -c conda-forge box2d-py
!pip install gymnasium tqdm torch torchvision

In [ ]:
!pip install pygame

In [ ]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

In [ ]:
!pip install pygame gymnasium[box2d]

In [ ]:
import gymnasium as gym

try:
    env = gym.make('CarRacing-v3')
    obs, info = env.reset()
    print("CarRacing-v3 환경 정상 작동!")
    env.close()
except Exception as e:
    print("환경 생성 실패:", e)

In [ ]:
import gymnasium as gym

# CarRacing-v3 환경 실행 (키보드 입력 포함)
env = gym.make("CarRacing-v3", render_mode="human")
obs, info = env.reset()

done = False
score = 0

while not done:
    # action을 아무것도 전달하지 않으면, 창에서 키보드 입력이 작동함
    action = env.action_space.sample()  # 무시됨, 직접 조작 가능!
    obs, reward, terminated, truncated, info = env.step(action)
    score += reward
    done = terminated or truncated

env.close()
print(f"플레이 종료! 총 점수: {score}")


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("CarRacing-v3", render_mode="human")
obs, info = env.reset()

done = False
score = 0

while not done:
    # 키보드 컨트롤만 받고 싶을 때는 action을 np.array로 넘겨야 함!
    obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0, 0.0]))
    score += reward
    done = terminated or truncated

env.close()
print(f"플레이 종료! 총 점수: {score}")


In [ ]:
!pip install stable-baselines3[extra] pygame


In [ ]:
!python -c "import torch; print(torch.version.cuda)"


In [ ]:
!pip uninstall -y torch torchvision torchaudio


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [ ]:
import torch
print(torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 사용 불가")


In [ ]:
pip install -U stable-baselines3

In [ ]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('./ppo_carracing_tensorboard/manual_test')
for i in range(10):
    writer.add_scalar('test_scalar', i, i)
writer.close()

In [ ]:
from stable_baselines3 import PPO
import gymnasium as gym

env = gym.make("CarRacing-v3")

model = PPO(
    "CnnPolicy",
    env,
    verbose=2,
    tensorboard_log="./ppo_carracing_tensorboard/",
    device="cuda"  # ★★★★★ 핵심!
)

model.learn(total_timesteps=3000000)
model.save("ppo_carracing")
env.close()


In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import numpy as np

env = gym.make("CarRacing-v3", render_mode="human")
model = PPO.load("ppo_carracing")

obs, info = env.reset()
score = 0
done = False

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    score += reward
    done = terminated or truncated

env.close()
print(f"훈련된 에이전트 점수: {score}")
